# Arabic Audio Transcriber — Local Notebook

Transcribes Arabic (and mixed Arabic/English) audio using **Faster-Whisper** on your local machine.

**Supports:**
- A local audio/video file (MP3, MP4, WAV, FLAC, M4A, MKV …)
- A YouTube URL (audio-only MP3 or full MP4)
- Generate an **SRT subtitle file** from the transcript
- **Burn subtitles** into the video with ffmpeg

**Outputs** (written to an `outputs/<name>/` folder next to this notebook):
- `audio.mp3` or `video.mp4` — downloaded / used audio
- `<name>_transcript.txt` — plain text transcript
- `<name>_transcript_timestamps.txt` — transcript with `[MM:SS-MM:SS]` markers
- `<name>.srt` *(optional)* — subtitle file
- `<name>_captioned.mp4` *(optional)* — video with burned-in captions

---
**Prerequisites (one-time setup):**
- Python ≥ 3.11 with a virtualenv activated
- `pip install faster-whisper yt-dlp torch tqdm`
- PyTorch CUDA build (if using GPU): `pip install torch --index-url https://download.pytorch.org/whl/cu128`
- [ffmpeg](https://ffmpeg.org/download.html) installed and on your PATH

Run **Cell 1** to verify all dependencies are present, then work top to bottom.

# Prompt

Translate the attached transcript into English. Preserve the intended meaning and word choice as closely as possible.
This is a generated transcript, so it may contain minor errors. Use the surrounding context to correct obvious transcription errors, but do not make large interpretive changes. For context, this is Islamic material.

Format the output as a timestamped transcript, for example:
[00:00-00:05] Hello everyone.

Timestamp rules:
Keep the original timestamp blocks and their order. Do not invent new timestamps, remove timestamps, extend timestamps, shorten timestamps, or change the start/end time of any timestamp block.
The translated transcript must begin and end at exactly the same timestamps as the source chunk.
You may only adjust caption balance by moving translated words between adjacent existing timestamp blocks when the speech clearly continues across them. Do not create new time ranges.
If a source line is very short, keep it short unless moving nearby words into it is necessary for readability.
If a source line is very long, keep the same timestamp and translate it fully; do not split it into new timestamps.

Continuity:
This video is long, so I will provide it in 20-minute chunks. Treat each part as continuing from the previous part, not as an independent transcript. Preserve terminology consistently across all parts.


Output rules:
Do not include notes, explanations, headings, corrections, or commentary.
Do not include anything other than the translated transcript.
Preserve the original wording as much as possible.

REMINDER: PRESERVE ORIGINAL WORDING AND ORIGINAL TIMESTAMP BOUNDARIES.

In [1]:
# ── Cell 1: Verify dependencies ──────────────────────────────────────────────
import importlib, shutil, sys

missing = [pkg for pkg in ["faster_whisper", "yt_dlp", "torch", "tqdm"] if importlib.util.find_spec(pkg) is None]
if missing:
    print("Missing Python packages:", ", ".join(missing))
    print("Install with:  pip install", " ".join(p.replace('_', '-') for p in missing))
else:
    print("✓ Python packages OK")

if importlib.util.find_spec("hf_xet") is None:
    print("⚠ 'hf_xet' not installed — model downloads from Hugging Face will be slow.")
    print("  Fix:  pip install hf_xet   (then Kernel → Restart)")
else:
    print("✓ hf_xet OK — fast Hugging Face downloads enabled")

if shutil.which("ffmpeg"):
    import subprocess
    v = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True).stdout.splitlines()[0]
    print(f"✓ ffmpeg: {v}")
else:
    print("✗ ffmpeg not found — install from https://ffmpeg.org/download.html and add it to PATH")

import torch
print(f"Python : {sys.executable}")
print(f"torch  : {torch.__version__}")

if torch.cuda.is_available():
    print(f"✓ GPU  : {torch.cuda.get_device_name(0)}")
elif "+cpu" in torch.__version__:
    print("⚠ CPU-only PyTorch (+cpu) — transcription will use CPU.")
    print("  Fix (in the project .venv, with it activated):")
    print("    pip uninstall torch torchaudio -y")
    print("    pip install torch torchaudio --index-url https://download.pytorch.org/whl/cu126")
    print("  Then: Kernel → Restart, and re-run this cell.")
else:
    print("⚠ torch.cuda.is_available() is False — will use CPU.")
    print("  Run:  python check_cuda.py   from the project folder for details.")

✓ Python packages OK
✓ hf_xet OK — fast Hugging Face downloads enabled
✓ ffmpeg: ffmpeg version 4.3 Copyright (c) 2000-2020 the FFmpeg developers
Python : c:\Users\subha\anaconda3\envs\transcribe\python.exe
torch  : 2.5.1
✓ GPU  : NVIDIA GeForce RTX 3070


## ⚙️ Configuration

Edit the values in the cell below before running anything else.

| Setting | Options | Notes |
|---------|---------|-------|
| `MODEL_SIZE` | `tiny` `base` `small` `medium` `large-v2` | Larger = more accurate, slower |
| `LANGUAGE` | `"ar"`, `"en"`, `"fr"` … or `None` | `None` = auto-detect from first 30 s |
| `START_TIME` | `"8:00"`, `"1:25:00"`, `None` | `None` = from beginning |
| `END_TIME` | `"25:00"`, `None` | `None` = to end of file |

In [2]:
# ── Cell 2: Configuration ─────────────────────────────────────────────────────
import re
import torch
from pathlib import Path

MODEL_SIZE = "large-v3-turbo"   # tiny | base | small | medium | large-v2
LANGUAGE   = "ar"          # e.g. "ar", "en", "fr", or None to auto-detect

# Optional time range — set to None to transcribe the whole file
START_TIME = None          # e.g. "8:00" or "1:25:00"
END_TIME   = None          # e.g. "25:00"

# YouTube video download: max height in pixels (720, 1080, …). Used in Cell 3B.
MAX_VIDEO_HEIGHT = 720

# Output directory (created automatically)
OUTPUTS_DIR = Path("outputs")

# ── Helpers ───────────────────────────────────────────────────────────────────
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"
COMPUTE_TYPE = "float16" if DEVICE == "cuda" else "int8"

def _parse_ts(s):
    """Parse 'SS', 'M:SS', 'MM:SS', or 'H:MM:SS' into seconds (float)."""
    if s is None:
        return None
    parts = str(s).strip().split(":")
    nums  = [float(p) for p in parts]
    if len(nums) == 1: return nums[0]
    if len(nums) == 2: return nums[0] * 60 + nums[1]
    return nums[0] * 3600 + nums[1] * 60 + nums[2]

def _fmt_ts(t):
    """Format seconds as MM:SS or H:MM:SS."""
    s = int(round(t))
    h, r = divmod(s, 3600)
    m, sec = divmod(r, 60)
    return f"{h}:{m:02d}:{sec:02d}" if h else f"{m:02d}:{sec:02d}"

def _slugify(text: str, max_len: int = 60) -> str:
    """Filesystem-safe name (keeps letters/digits/Arabic, collapses spaces to _)."""
    text = text.strip().lower()
    text = re.sub(r"[^\w\s-]", "", text, flags=re.UNICODE)
    text = re.sub(r"[\s_-]+", "_", text)
    return text.strip("_")[:max_len] or "output"

def _unique_dir(base: Path) -> Path:
    """Return base if unused, otherwise base_2, base_3, …"""
    if not base.exists():
        return base
    i = 2
    while True:
        candidate = base.parent / f"{base.name}_{i}"
        if not candidate.exists():
            return candidate
        i += 1

start_sec = _parse_ts(START_TIME)
end_sec   = _parse_ts(END_TIME)

print(f"Model   : {MODEL_SIZE}  |  Device: {DEVICE}  |  Compute: {COMPUTE_TYPE}")
print(f"Language: {LANGUAGE or 'auto-detect'}")
if start_sec is not None or end_sec is not None:
    s = _fmt_ts(start_sec or 0)
    e = _fmt_ts(end_sec) if end_sec is not None else "end"
    print(f"Range   : {s} → {e}")
else:
    print("Range   : full file")

Model   : large-v3-turbo  |  Device: cuda  |  Compute: float16
Language: ar
Range   : full file


## 📦 Model cache

Downloads `MODEL_SIZE` from Hugging Face **once** and reuses the local cache on every later run — no re-download, no network call, unless you explicitly ask to check for updates.

In [3]:
# ── Cell 2B: Model cache — download once, reuse offline ──────────────────────
#
# First run of a new MODEL_SIZE: downloads the model (this is the slow part —
# install `hf_xet` per Cell 1 to speed it up).
# Every later run: loads straight from the local Hugging Face cache with
# local_files_only=True, so there is zero network traffic and zero delay.
#
# Set CHECK_FOR_MODEL_UPDATES = True and re-run this cell to check Hugging Face
# for a newer revision of the model (only re-downloads if something changed).

from pathlib import Path
from faster_whisper.utils import download_model

CHECK_FOR_MODEL_UPDATES = False


def _is_complete(model_dir) -> bool:
    """local_files_only=True happily returns a snapshot dir even if the big
    weights file never finished downloading (e.g. a previous run was
    interrupted). Verify model.bin actually resolves before trusting it."""
    p = Path(model_dir) / "model.bin"
    return p.exists() and p.stat().st_size > 0


MODEL_PATH = None
if not CHECK_FOR_MODEL_UPDATES:
    try:
        candidate = download_model(MODEL_SIZE, local_files_only=True)
        if _is_complete(candidate):
            MODEL_PATH = candidate
            print(f"✓ Model '{MODEL_SIZE}' ready (local cache): {MODEL_PATH}")
        else:
            print(f"'{MODEL_SIZE}' is cached but incomplete (missing/empty model.bin) — will re-download.")
    except Exception:
        pass

if MODEL_PATH is None:
    print(f"Downloading '{MODEL_SIZE}' (one-time — this is the slow part; hf_xet speeds it up)…")
    MODEL_PATH = download_model(MODEL_SIZE, local_files_only=False)
    if not _is_complete(MODEL_PATH):
        raise RuntimeError(f"Download finished but model.bin is still missing/empty in {MODEL_PATH}")
    print(f"✓ Downloaded and cached: {MODEL_PATH}")

C:\Users\subha\AppData\Roaming\Python\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✓ Model 'large-v3-turbo' ready (local cache): C:\Users\subha\.cache\huggingface\hub\models--mobiuslabsgmbh--faster-whisper-large-v3-turbo\snapshots\0a363e9161cbc7ed1431c9597a8ceaf0c4f78fcf


## 📥 Input — run **one** of the two cells below

**Option A** — point at a local file  
**Option B** — paste a YouTube URL

In [4]:
# ── Cell 3A: Use a local file ─────────────────────────────────────────────────
# Run this cell OR Cell 3B — not both.

INPUT_FILE = r"test.mp3"   # ← edit this path

VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}

src = Path(INPUT_FILE)
if not src.exists():
    raise FileNotFoundError(f"File not found: {src}")

STEM = _slugify(src.stem)
OUTPUT_DIR = _unique_dir(OUTPUTS_DIR / STEM)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VIDEO_PATH = src if src.suffix.lower() in VIDEO_EXTS else None
AUDIO_PATH = src   # faster-whisper reads any ffmpeg-compatible format directly

print(f"Input      : {src}  ({src.stat().st_size / 1e6:.1f} MB)")
print(f"Output dir : {OUTPUT_DIR}")
if VIDEO_PATH:
    print("Video detected — captioning available after transcription.")
else:
    print("Audio-only input — captioning step will be skipped.")

Input      : test.mp3  (0.9 MB)
Output dir : outputs\test
Audio-only input — captioning step will be skipped.


In [5]:
# ── Cell 4: Transcribe ────────────────────────────────────────────────────────
from faster_whisper import WhisperModel

print(f"Loading Faster-Whisper '{MODEL_SIZE}' on {DEVICE} ({COMPUTE_TYPE}) …")
model = WhisperModel(MODEL_PATH, device=DEVICE, compute_type=COMPUTE_TYPE)

use_clip   = start_sec is not None or end_sec is not None
clip_start = start_sec or 0.0

if not use_clip:
    clip_timestamps = "0"
    vad_filter = True
elif end_sec is None:
    clip_timestamps = str(clip_start)
    vad_filter = False
else:
    clip_timestamps = f"{clip_start},{float(end_sec)}"
    vad_filter = False

print(f"Transcribing {AUDIO_PATH.name} …")
segments, info = model.transcribe(
    str(AUDIO_PATH),
    language=LANGUAGE,
    beam_size=5,
    temperature=0.0,
    condition_on_previous_text=False,
    vad_filter=vad_filter,
    clip_timestamps=clip_timestamps,

    # Helpful for long-form transcription
    word_timestamps=False,
    compression_ratio_threshold=2.4,
    log_prob_threshold=-1.0,
    no_speech_threshold=0.6
)

duration = getattr(info, "duration", None)
if duration:
    print(f"Audio length : {_fmt_ts(duration)}")
if use_clip:
    eff_end = min(end_sec, duration) if (end_sec and duration) else (end_sec or duration or 0)
    print(f"Transcribing : {_fmt_ts(clip_start)} → {_fmt_ts(eff_end)}")

texts, timed_lines = [], []
for seg in segments:
    text = seg.text.strip()
    if not text:
        continue
    seg_start = float(seg.start or 0)
    seg_end   = float(seg.end   or 0)
    line = f"[{_fmt_ts(seg_start)}-{_fmt_ts(seg_end)}] {text}"
    texts.append(text)
    timed_lines.append(line)
    print(line)

full_text  = " ".join(texts)
timed_text = "\n".join(timed_lines)
print("\n✓ Transcription complete.")

Loading Faster-Whisper 'large-v3-turbo' on cuda (float16) …
Transcribing test.mp3 …
Audio length : 00:53
[00:00-00:04] أحكم أقرأ القرآن عند جنابة
[00:04-00:07] عندي إشكال يعني عند النوم
[00:07-00:11] وأنا في جنابة كيف أقرأ القرآن
[00:11-00:15] كيف أقرأ آخر آية أذكار النوم يعني
[00:15-00:21] نعم أما الجنوب فبتفق أربعة وهو قول عمر وعلي
[00:21-00:26] ثبت عن علي عنده بشيبة وعمر كما في خلافيات البيهقي
[00:26-00:28] قال علي أما الجنوب فلا ولو آية
[00:28-00:32] وهذا بخلاف ابن عباس فقد علق عن البخاري
[00:32-00:33] عباس الجنوب يقرأ القرآن
[00:33-00:36] وهو قول جماعة كالبخاري وغيره
[00:36-00:37] والصواب أن الجنوب لا يقرأ القرآن
[00:37-00:40] أما أذكار النوم فهي قسمان
[00:40-00:43] القسم الأول أذكار والقسم الثاني قرآن
[00:43-00:46] كآية الكرسي وآخر آيتين من البقرة إلى آخر
[00:46-00:47] فلا يقرأها الجنوب
[00:47-00:50] لذا يستحب الجنوب أن لا ينام جنوبا
[00:50-00:53] لأسباب منها أن يقرأ هذه الأذكار نعم

✓ Transcription complete.


In [6]:
# ── Cell 5: Save transcripts ──────────────────────────────────────────────────
plain_path = OUTPUT_DIR / f"{STEM}_transcript.txt"
timed_path = OUTPUT_DIR / f"{STEM}_transcript_timestamps.txt"

plain_path.write_text(full_text,  encoding="utf-8")
timed_path.write_text(timed_text, encoding="utf-8")

print(f"Plain transcript   → {plain_path}")
print(f"Timed transcript   → {timed_path}  ({len(timed_lines)} segments)")
print()
print("──── Preview (first 2 000 chars) ────")
print(timed_text[:2000])
if len(timed_text) > 2000:
    print(f"… ({len(timed_text) - 2000} more characters)")

Plain transcript   → outputs\test\test_transcript.txt
Timed transcript   → outputs\test\test_transcript_timestamps.txt  (17 segments)

──── Preview (first 2 000 chars) ────
[00:00-00:04] أحكم أقرأ القرآن عند جنابة
[00:04-00:07] عندي إشكال يعني عند النوم
[00:07-00:11] وأنا في جنابة كيف أقرأ القرآن
[00:11-00:15] كيف أقرأ آخر آية أذكار النوم يعني
[00:15-00:21] نعم أما الجنوب فبتفق أربعة وهو قول عمر وعلي
[00:21-00:26] ثبت عن علي عنده بشيبة وعمر كما في خلافيات البيهقي
[00:26-00:28] قال علي أما الجنوب فلا ولو آية
[00:28-00:32] وهذا بخلاف ابن عباس فقد علق عن البخاري
[00:32-00:33] عباس الجنوب يقرأ القرآن
[00:33-00:36] وهو قول جماعة كالبخاري وغيره
[00:36-00:37] والصواب أن الجنوب لا يقرأ القرآن
[00:37-00:40] أما أذكار النوم فهي قسمان
[00:40-00:43] القسم الأول أذكار والقسم الثاني قرآن
[00:43-00:46] كآية الكرسي وآخر آيتين من البقرة إلى آخر
[00:46-00:47] فلا يقرأها الجنوب
[00:47-00:50] لذا يستحب الجنوب أن لا ينام جنوبا
[00:50-00:53] لأسباب منها أن يقرأ هذه الأذكار نعم


## 🎬 Subtitles (optional)

Two follow-up steps you can run after transcription:

1. **Cell 6** — Convert the timestamped transcript to an **SRT** file.
2. **Cell 7** — Burn the SRT into the video with ffmpeg → `*_captioned.mp4`.

Step 2 requires a video. You have one if you:
- Used a local video file in **Cell 3A**, or  
- Used `DOWNLOAD_VIDEO = True` in **Cell 3B**.

In [7]:
# ── Cell 6: Generate SRT ──────────────────────────────────────────────────────
#
# SRT_SOURCE controls where the input comes from:
#   "session" — use timed_text from Cell 4 (this session)
#   "file"    — read from an existing transcript file on disk

import re

SRT_SOURCE = "file"   # "session" or "file"

# Only used when SRT_SOURCE = "file"
TRANSCRIPT_FILE = "test.txt"

if SRT_SOURCE == "file":
    src_path    = Path(TRANSCRIPT_FILE)
    source_text = src_path.read_text(encoding="utf-8")
    srt_stem    = src_path.stem
    srt_dir     = src_path.parent
else:
    if "timed_text" not in dir():
        raise RuntimeError("No transcript in this session — run Cell 4 first, or set SRT_SOURCE = 'file'.")
    source_text = timed_text
    srt_stem    = STEM
    srt_dir     = OUTPUT_DIR

# ── Parse ─────────────────────────────────────────────────────────────────────
pattern = re.compile(r"\[(\d+(?::\d{2})+)\s*-\s*(\d+(?::\d{2})+)\]\s*(.*)")

def _ts_to_seconds(ts):
    nums = [int(p) for p in ts.split(":")]
    if len(nums) == 2: return nums[0] * 60 + nums[1]
    if len(nums) == 3: return nums[0] * 3600 + nums[1] * 60 + nums[2]
    raise ValueError(f"Cannot parse: {ts!r}")

def _seconds_to_srt(t):
    ms = int(round(t * 1000))
    h,  rem = divmod(ms, 3_600_000)
    m,  rem = divmod(rem,    60_000)
    s,  ms  = divmod(rem,     1_000)
    return f"{h:02d}:{m:02d}:{s:02d},{ms:03d}"

entries, counter = [], 1
for line in source_text.splitlines():
    mat = pattern.match(line.strip())
    if not mat:
        continue
    start_raw, end_raw, text = mat.groups()
    text = text.strip()
    if not text:
        continue
    entries.append(f"{counter}\n{_seconds_to_srt(_ts_to_seconds(start_raw))} --> {_seconds_to_srt(_ts_to_seconds(end_raw))}\n{text}\n")
    counter += 1

if not entries:
    raise RuntimeError("No [MM:SS-MM:SS] lines found. Check the transcript format.")

SRT_PATH = srt_dir / f"{srt_stem}.srt"
SRT_PATH.write_text("\n".join(entries), encoding="utf-8")
print(f"✓ Wrote {SRT_PATH}  ({counter - 1} captions)")
print()
print("──── Preview (first 1 000 chars) ────")
print(SRT_PATH.read_text(encoding="utf-8")[:1000])

✓ Wrote test.srt  (17 captions)

──── Preview (first 1 000 chars) ────
1
00:00:00,000 --> 00:00:04,000
Is it permissible for me to recite the Qur'an while in a state of janabah?

2
00:00:04,000 --> 00:00:07,000
I have a question regarding going to sleep.

3
00:00:07,000 --> 00:00:11,000
While I am in a state of janabah, how can I recite the Qur'an?

4
00:00:11,000 --> 00:00:15,000
How can I recite the last verses and the bedtime adhkar?

5
00:00:15,000 --> 00:00:21,000
Yes. As for the person in a state of janabah, four companions agreed on this, and it is the view of 'Umar and 'Ali.

6
00:00:21,000 --> 00:00:26,000
It is authentically reported from 'Ali in Ibn Abi Shaybah, and from 'Umar as found in al-Bayhaqi's *al-Khilafiyyat*.

7
00:00:26,000 --> 00:00:28,000
'Ali said: "As for the person in janabah, not even a single verse."

8
00:00:28,000 --> 00:00:32,000
This is contrary to Ibn 'Abbas. Al-Bukhari cited from him in a suspended narration

9
00:00:32,000 --> 00:00:33,000
that a per

In [42]:
# ── Cell 7: Burn captions into the video with ffmpeg ─────────────────────────
#
# Set paths below. On Windows use a raw string (r"...") OR forward slashes:
#   r"C:\Users\you\outputs\my_lecture\my_lecture.mp4"
#   "C:/Users/you/outputs/my_lecture/my_lecture.mp4"
# Leave as None to use VIDEO_PATH / SRT_PATH from Cells 3–6.

import shutil
import subprocess
from pathlib import Path

MANUAL_VIDEO_PATH = "10.mp4"   # e.g. r"C:\...\outputs\my_lecture\my_lecture.mp4"
MANUAL_SRT_PATH   = "10.srt"   # e.g. r"C:\...\outputs\my_lecture\my_lecture.srt"

# Leave None to auto-pick a full ffmpeg build (conda/pip ffmpeg often lacks libx264).
FFMPEG_PATH = None   # e.g. r"C:\ffmpeg\bin\ffmpeg.exe"

_VIDEO_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v"}
_SRT_EXTS   = {".srt"}


def _ffmpeg_candidates() -> list[Path]:
    seen: set[str] = set()
    candidates: list[Path] = []

    def _add(path: Path | None) -> None:
        if path is None:
            return
        key = str(path.resolve()).lower()
        if key not in seen:
            seen.add(key)
            candidates.append(path)

    if FFMPEG_PATH:
        _add(Path(str(FFMPEG_PATH).strip().strip('"').strip("'")).expanduser())

    # Prefer known full builds over PATH. A notebook kernel can retain an old
    # PATH entry even after FFmpeg has been upgraded.
    for pattern in (
        r"C:\ffmpeg\bin\ffmpeg.exe",
        r"C:\Program Files\ffmpeg\bin\ffmpeg.exe",
    ):
        _add(Path(pattern))

    winget_root = Path.home() / "AppData/Local/Microsoft/WinGet/Packages"
    if winget_root.is_dir():
        for exe in sorted(winget_root.glob("Gyan.FFmpeg*/ffmpeg-*/bin/ffmpeg.exe"), reverse=True):
            _add(exe)

    which = shutil.which("ffmpeg")
    if which:
        _add(Path(which))

    return candidates


def _available_encoders(ffmpeg: Path) -> set[str]:
    """Return exact encoder names from `ffmpeg -encoders`."""
    probe = subprocess.run(
        [str(ffmpeg), "-hide_banner", "-encoders"],
        capture_output=True,
        text=True,
    )
    if probe.returncode != 0:
        return set()

    names: set[str] = set()
    for line in probe.stdout.splitlines():
        parts = line.split()
        # Encoder rows are: six capability flags, encoder name, description.
        if len(parts) >= 2 and len(parts[0]) == 6 and parts[0][0] in "VAS":
            names.add(parts[1])
    return names


def _resolve_ffmpeg() -> Path:
    for exe in _ffmpeg_candidates():
        if not exe.is_file():
            continue
        encoders = _available_encoders(exe)
        if not ({"libx264", "h264_nvenc", "h264_mf"} & encoders):
            continue

        version_lines = subprocess.run(
            [str(exe), "-version"],
            capture_output=True,
            text=True,
        ).stdout.splitlines()
        print(f"ffmpeg : {exe}")
        if version_lines:
            print(f"         {version_lines[0]}")
        return exe

    raise RuntimeError(
        "No ffmpeg with an H.264 encoder found.\n"
        "Install a full build (e.g. winget install Gyan.FFmpeg) or set FFMPEG_PATH "
        "to something like r\"C:\\ffmpeg\\bin\\ffmpeg.exe\"."
    )


def _video_encode_args(ffmpeg: Path, crf: int, preset: str) -> tuple[str, list[str]]:
    encoders = _available_encoders(ffmpeg)

    if "libx264" in encoders:
        return "libx264", ["-c:v", "libx264", "-crf", str(crf), "-preset", preset]
    if "h264_nvenc" in encoders:
        return "h264_nvenc", ["-c:v", "h264_nvenc", "-rc", "vbr", "-cq", str(crf)]
    if "h264_mf" in encoders:
        quality = max(10, min(100, 110 - crf * 3))
        return "h264_mf", ["-c:v", "h264_mf", "-rate_control", "quality", "-quality", str(quality)]

    raise RuntimeError("ffmpeg has no usable H.264 encoder (libx264, h264_nvenc, h264_mf).")


def _resolve_file(
    manual: str | Path | None,
    session_var: str,
    allowed_exts: set[str],
) -> Path:
    """Resolve a file path from manual input or an earlier notebook variable."""
    if manual is not None and str(manual).strip():
        p = Path(str(manual).strip().strip('"').strip("'")).expanduser()
        source = "MANUAL_*_PATH"
    elif session_var in dir() and globals().get(session_var) is not None:
        p = Path(globals()[session_var]).expanduser()
        source = session_var
    else:
        raise FileNotFoundError(
            f"No path for {session_var}. Set MANUAL_{session_var} above or run Cells 3–6."
        )

    # Catch broken Windows paths like "outputs\rayyis..." where \r is a control char
    if any(ord(c) < 32 for c in str(p)):
        raise ValueError(
            f"Invalid characters in path from {source}: {p!r}\n"
            "On Windows, use a raw string:  r\"C:\\\\path\\\\video.mp4\"\n"
            "Or forward slashes:        \"C:/path/video.mp4\""
        )

    if p.is_dir():
        matches = [f for f in p.rglob("*") if f.is_file() and f.suffix.lower() in allowed_exts]
        if len(matches) == 1:
            p = matches[0]
        elif matches:
            listing = "\n".join(f"  {m}" for m in sorted(matches)[:15])
            raise FileNotFoundError(
                f"{source} is a folder, not a file: {p}\n"
                f"Set MANUAL_*_PATH to one of these files:\n{listing}"
            )
        else:
            raise FileNotFoundError(f"{source} is a folder with no matching files: {p}")

    if p.suffix.lower() not in allowed_exts:
        raise FileNotFoundError(
            f"Expected {allowed_exts}, got {p.suffix!r} for {source}: {p}"
        )

    if not p.is_file():
        raise FileNotFoundError(
            f"File not found ({source}):\n  {p}\n"
            f"  (repr: {p!r})\n"
            "Check the path. Use r\"...\" on Windows or forward slashes."
        )

    return p.resolve()

_vid = _resolve_file(MANUAL_VIDEO_PATH, "VIDEO_PATH", _VIDEO_EXTS)
_srt = _resolve_file(MANUAL_SRT_PATH, "SRT_PATH", _SRT_EXTS)

print(f"Video : {_vid}")
print(f"SRT   : {_srt}")

# Subtitle style
SUBTITLE_STYLE = (
    "FontName=Arial,"
    "FontSize=18,"
    "PrimaryColour=&H00FFFFFF,"    # white text (AABBGGRR)
    "BackColour=&H00000000,"       # ~50% transparent black box
    "BorderStyle=3,"               # 3 = opaque background box
    "MarginV=40,"
    "Alignment=2"                  # bottom center
)

# Encode quality
VIDEO_CRF    = 14       # lower = better quality (try 16–22)
VIDEO_PRESET = "medium" # ultrafast … veryslow

# ffmpeg's subtitles filter breaks on Windows absolute paths (C: is parsed as an option).
# Run in the video folder with a simple relative .srt filename instead.
work_dir = _vid.parent
OUT_VIDEO = work_dir / f"{_vid.stem}_captioned.mp4"
_burn_srt = work_dir / "_burn_subs.srt"
shutil.copy2(_srt, _burn_srt)

filter_arg = f"subtitles=_burn_subs.srt:force_style='{SUBTITLE_STYLE}'"

_ffmpeg = _resolve_ffmpeg()
_encoder, _encode_args = _video_encode_args(_ffmpeg, VIDEO_CRF, VIDEO_PRESET)

print(f"Burning {_srt.name} → {OUT_VIDEO.name}")
print(f"Working directory: {work_dir}")
print(f"Encode : {_encoder} crf={VIDEO_CRF} preset={VIDEO_PRESET}")

result = subprocess.run(
    [
        str(_ffmpeg), "-y",
        "-i", _vid.name,
        "-vf", filter_arg,
        *_encode_args,
        "-c:a", "copy",
        OUT_VIDEO.name,
    ],
    cwd=work_dir,
    capture_output=True,
    text=True,
)

if result.returncode != 0:
    print("ffmpeg failed:\n", result.stderr[-4000:])
    raise RuntimeError(f"ffmpeg exited with code {result.returncode}")

_burn_srt.unlink(missing_ok=True)
print(f"✓ Wrote {OUT_VIDEO}  ({OUT_VIDEO.stat().st_size / 1e6:.1f} MB)")

Video : C:\Users\subha\Desktop\transcribe\youtube-translator\10.mp4
SRT   : C:\Users\subha\Desktop\transcribe\youtube-translator\10.srt
ffmpeg : C:\ffmpeg\bin\ffmpeg.exe
         ffmpeg version 7.1.1-full_build-www.gyan.dev Copyright (c) 2000-2025 the FFmpeg developers
Burning 10.srt → 10_captioned.mp4
Working directory: C:\Users\subha\Desktop\transcribe\youtube-translator
Encode : libx264 crf=14 preset=medium
✓ Wrote C:\Users\subha\Desktop\transcribe\youtube-translator\10_captioned.mp4  (202.6 MB)


## 🎞️ Final video from audio (optional)

**Cell 8** builds a standalone video for audio-only lectures:

1. `Intro.mp4` plays first (with its own audio).
2. `Background.mp4` loops under the main audio until it ends — the last loop plays out fully instead of being cut to the audio.
3. The clip ends with a fade to black.

The main audio starts right after the intro, and the SRT captions are automatically shifted by the intro duration so they stay in sync.

In [10]:
# ── Cell 8: Intro + looping background + audio + subtitles → final video ─────
#
# Builds:  Intro.mp4  →  Background.mp4 looped under the main audio  →  fade to black.
# The last background loop plays out fully (video runs a little past the audio),
# and the SRT is shifted by the intro duration so captions stay in sync.
#
# Reuses _resolve_ffmpeg/_video_encode_args/_resolve_file from Cell 7 when that
# cell has been run; otherwise falls back to a simple ffmpeg lookup.

import re
import shutil
import subprocess
from math import ceil
from pathlib import Path

FINAL_AUDIO_PATH = "test.mp3"           # main audio (any ffmpeg-readable audio/video file)
FINAL_SRT_PATH   = "test.srt"           # captions, timed relative to the main audio
INTRO_PATH       = "Intro.mp4"        # plays first, with its own audio
BACKGROUND_PATH  = "Background.mp4"   # loops under the main audio

FADE_SECONDS = 1.5     # fade-to-black duration at the very end
TARGET_W, TARGET_H, TARGET_FPS = 1920, 1080, 24

# Encode quality (same knobs as Cell 7)
VIDEO_CRF    = 14
VIDEO_PRESET = "medium"
AUDIO_BITRATE = "192k"

SUBTITLE_STYLE = (
    "FontName=Arial,"
    "FontSize=16,"
    "PrimaryColour=&H00FFFFFF,"
    "BackColour=&H00000000,"
    "BorderStyle=3,"
    "MarginV=40,"
    "Alignment=2"
)

# ── Resolve inputs ────────────────────────────────────────────────────────────
_nb_dir = Path.cwd()

def _final_resolve(p: str | Path, exts: set[str], label: str) -> Path:
    p = Path(str(p).strip().strip('"').strip("'")).expanduser()
    if not p.is_absolute():
        p = _nb_dir / p
    if not p.is_file():
        raise FileNotFoundError(f"{label} not found: {p}")
    if p.suffix.lower() not in exts:
        raise ValueError(f"{label}: expected {exts}, got {p.suffix!r}: {p}")
    return p.resolve()

_MEDIA_EXTS = {".mp4", ".mkv", ".mov", ".webm", ".avi", ".m4v", ".mp3", ".wav", ".m4a", ".flac", ".aac", ".ogg", ".opus"}
_audio = _final_resolve(FINAL_AUDIO_PATH, _MEDIA_EXTS, "FINAL_AUDIO_PATH")
_srt   = _final_resolve(FINAL_SRT_PATH, {".srt"}, "FINAL_SRT_PATH")
_intro = _final_resolve(INTRO_PATH, {".mp4", ".mkv", ".mov", ".webm", ".m4v"}, "INTRO_PATH")
_bg    = _final_resolve(BACKGROUND_PATH, {".mp4", ".mkv", ".mov", ".webm", ".m4v"}, "BACKGROUND_PATH")

# ── Resolve ffmpeg / ffprobe ──────────────────────────────────────────────────
if "_resolve_ffmpeg" in globals():
    _ffmpeg = _resolve_ffmpeg()
    _encoder, _encode_args = _video_encode_args(_ffmpeg, VIDEO_CRF, VIDEO_PRESET)
else:
    for _cand in [globals().get("FFMPEG_PATH"), r"C:\ffmpeg\bin\ffmpeg.exe", shutil.which("ffmpeg")]:
        if _cand and Path(_cand).is_file():
            _ffmpeg = Path(_cand)
            break
    else:
        raise RuntimeError("ffmpeg not found — run Cell 7 first or set FFMPEG_PATH.")
    print(f"ffmpeg : {_ffmpeg}")
    _encoder, _encode_args = "libx264", ["-c:v", "libx264", "-crf", str(VIDEO_CRF), "-preset", VIDEO_PRESET]

_ffprobe = _ffmpeg.with_name("ffprobe.exe" if _ffmpeg.suffix else "ffprobe")
if not _ffprobe.is_file():
    _ffprobe = Path(shutil.which("ffprobe") or "")
    if not _ffprobe.is_file():
        raise RuntimeError("ffprobe not found next to ffmpeg or on PATH.")

def _duration(path: Path) -> float:
    r = subprocess.run(
        [str(_ffprobe), "-v", "error", "-show_entries", "format=duration",
         "-of", "default=noprint_wrappers=1:nokey=1", str(path)],
        capture_output=True, text=True,
    )
    if r.returncode != 0 or not r.stdout.strip():
        raise RuntimeError(f"ffprobe failed for {path}:\n{r.stderr[-1000:]}")
    return float(r.stdout.strip())

intro_dur = _duration(_intro)
bg_dur    = _duration(_bg)
audio_dur = _duration(_audio)

# Last background loop plays out fully instead of being trimmed to the audio.
bg_loops  = max(1, ceil(audio_dur / bg_dur))
total_dur = intro_dur + bg_loops * bg_dur
fade_dur  = min(FADE_SECONDS, total_dur)

print(f"Intro      : {_intro.name}  ({intro_dur:.2f}s)")
print(f"Background : {_bg.name}  ({bg_dur:.2f}s × {bg_loops} loops = {bg_loops * bg_dur:.2f}s)")
print(f"Main audio : {_audio.name}  ({audio_dur:.2f}s, starts at {intro_dur:.2f}s)")
print(f"Total      : {total_dur:.2f}s  (fade to black over last {fade_dur:.1f}s)")

# ── Shift SRT by the intro duration ───────────────────────────────────────────
_srt_time = re.compile(r"(\d{2,}):(\d{2}):(\d{2}),(\d{3})")

def _shift_srt_text(text: str, offset_sec: float) -> str:
    off_ms = int(round(offset_sec * 1000))
    def _repl(m):
        h, mnt, s, ms = (int(g) for g in m.groups())
        t = h * 3_600_000 + mnt * 60_000 + s * 1_000 + ms + off_ms
        h, rem = divmod(t, 3_600_000)
        mnt, rem = divmod(rem, 60_000)
        s, ms = divmod(rem, 1_000)
        return f"{h:02d}:{mnt:02d}:{s:02d},{ms:03d}"
    return _srt_time.sub(_repl, text)

# subtitles filter chokes on Windows absolute paths → work in the audio's folder
work_dir  = _audio.parent
OUT_FINAL = work_dir / f"{_audio.stem}_final.mp4"
_shifted_srt = work_dir / "_final_subs.srt"
_shifted_srt.write_text(_shift_srt_text(_srt.read_text(encoding="utf-8"), intro_dur), encoding="utf-8")

# ── Build and run ffmpeg ──────────────────────────────────────────────────────
_scale = (
    f"scale={TARGET_W}:{TARGET_H}:force_original_aspect_ratio=decrease,"
    f"pad={TARGET_W}:{TARGET_H}:-1:-1:color=black,setsar=1,fps={TARGET_FPS}"
)
filter_complex = (
    f"[0:v]{_scale}[iv];"
    f"[1:v]{_scale}[bv];"
    f"[iv][bv]concat=n=2:v=1:a=0[v0];"
    f"[v0]subtitles=_final_subs.srt:force_style='{SUBTITLE_STYLE}'[v1];"
    f"[v1]fade=t=out:st={total_dur - fade_dur:.3f}:d={fade_dur:.3f}[vout];"
    f"[0:a]aresample=48000,aformat=channel_layouts=stereo[ia];"
    f"[2:a]aresample=48000,aformat=channel_layouts=stereo[ma];"
    f"[ia][ma]concat=n=2:v=0:a=1[a0];"
    f"[a0]apad=whole_dur={total_dur:.3f}[aout]"
)

cmd = [
    str(_ffmpeg), "-y",
    "-i", str(_intro),
    "-stream_loop", str(bg_loops - 1), "-i", str(_bg),
    "-i", str(_audio),
    "-filter_complex", filter_complex,
    "-map", "[vout]", "-map", "[aout]",
    *_encode_args,
    "-c:a", "aac", "-b:a", AUDIO_BITRATE,
    "-t", f"{total_dur:.3f}",
    "-movflags", "+faststart",
    OUT_FINAL.name,
]

print(f"\nEncoding with {_encoder} → {OUT_FINAL.name} …")
result = subprocess.run(cmd, cwd=work_dir, capture_output=True, text=True)

if result.returncode != 0:
    print("ffmpeg failed:\n", result.stderr[-4000:])
    raise RuntimeError(f"ffmpeg exited with code {result.returncode}")

_shifted_srt.unlink(missing_ok=True)
print(f"✓ Wrote {OUT_FINAL}  ({OUT_FINAL.stat().st_size / 1e6:.1f} MB)")

ffmpeg : C:\ffmpeg\bin\ffmpeg.exe
Intro      : Intro.mp4  (10.01s)
Background : Background.mp4  (8.17s × 7 loops = 57.19s)
Main audio : test.mp3  (53.40s, starts at 10.01s)
Total      : 67.20s  (fade to black over last 1.5s)

Encoding with libx264 → test_final.mp4 …
✓ Wrote C:\Users\subha\Desktop\transcribe\youtube-translator\test_final.mp4  (54.6 MB)
